<a href="https://colab.research.google.com/github/dontasksteven/IS4487/blob/main/Assignments/assignment_11_PetersonSteven.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [4]:
url="/content/cleaned_airbnb_data.csv"
df = pd.read_csv(url)

In [9]:
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,257869,https://www.airbnb.com/rooms/257869,20250613040024,2025-06-13,previous scrape,House w/ Pool in Cultural District/close2downtown,No STR's! <br />Be my guest and enjoy everythi...,Very quiet neighborhood. <br />Tree lined stre...,https://a0.muscache.com/pictures/18939849/1e85...,1356081,...,4.97,4.96,4.92,NaN,f,1,1,0,0,2.31
1,299748,https://www.airbnb.com/rooms/299748,20250613040024,2025-06-13,previous scrape,Rosy's Guest House,"Quiet, ground floor studio apartment in a gorg...",Gorgeous historic homes with tree lined sidew...,https://a0.muscache.com/pictures/4eead819-7ad8...,1544441,...,4.98,4.96,4.96,NaN,f,1,1,0,0,2.20
2,912371,https://www.airbnb.com/rooms/912371,20250613040024,2025-06-13,city scrape,Stay in Ft.Worth Cultural District,"Quiet, studio, upstairs garage apartment surro...","This is a beautiful, safe, quiet neighborhood ...",https://a0.muscache.com/pictures/13577479/7098...,4895584,...,4.96,4.95,4.83,NaN,f,1,1,0,0,2.15
3,1885626,https://www.airbnb.com/rooms/1885626,20250613040024,2025-06-13,city scrape,Private bedroom in spacious home,"If you're like us, friendly Canadians, you jus...",This neighbourhood is very quiet and family or...,https://a0.muscache.com/pictures/37169857/d6f1...,7892516,...,4.98,4.91,4.96,NaN,f,1,0,1,0,0.68
4,2130492,https://www.airbnb.com/rooms/2130492,20250613040024,2025-06-13,city scrape,"Charming ""attic room"" by TCU campus",Our cozy 2-story home overlooks the TCU stadiu...,"This is a wonderfully friendly old, yet lively...",https://a0.muscache.com/pictures/54799567/0f33...,826752,...,5.00,4.98,5.00,NaN,f,2,0,2,0,0.32


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2047 entries, 0 to 2046
Data columns (total 75 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            2047 non-null   int64  
 1   listing_url                                   2047 non-null   object 
 2   scrape_id                                     2047 non-null   int64  
 3   last_scraped                                  2047 non-null   object 
 4   source                                        2047 non-null   object 
 5   name                                          2047 non-null   object 
 6   description                                   2019 non-null   object 
 7   neighborhood_overview                         899 non-null    object 
 8   picture_url                                   2047 non-null   object 
 9   host_id                                       2047 non-null   i

### ✍️ Your Response: 🔧
1. The dataset includes a large variety of info regarding Airbnb data from location, info about the actual property, and additional data regarding host information.

2. Our dataset has 2047 (2046 excluding headers) rows and 75 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [14]:
columns_to_drop = ['id', 'name', 'description', 'scrape_id', 'listing_url', 'neighborhood_overview', 'picture_url', 'last_scraped', 'source', 'host_url', 'neighborhood_cleansed', 'calendar_last_scraped', 'license', 'calendar_updated']

# Check which of these columns actually exist in the DataFrame
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

if existing_columns_to_drop:
    df.drop(columns=existing_columns_to_drop, inplace=True)
    print(f"Dropped columns: {existing_columns_to_drop}")
else:
    print("No specified columns found to drop.")

df.head()
df.info()

Dropped columns: ['license', 'calendar_updated']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2047 entries, 0 to 2046
Data columns (total 62 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_id                                       2047 non-null   int64  
 1   host_name                                     2043 non-null   object 
 2   host_since                                    2043 non-null   object 
 3   host_location                                 1476 non-null   object 
 4   host_response_time                            1798 non-null   object 
 5   host_response_rate                            1798 non-null   object 
 6   host_acceptance_rate                          1852 non-null   object 
 7   host_is_superhost                             1969 non-null   object 
 8   host_thumbnail_url                            2043 non-null   object 
 9   host_picture_u

### ✍️ Your Response: 🔧
1. The majority of the columns that I dropped were either columns that the meaning was rather ambiguous or had to decipher, such as neighborhood_cleansed. Then the other columns I dropped were those specifically related to URL information and other "long" descriptive information.

2. The biggest risk we see when dropping these columns is potentially losing large amounts of identifiable information. If we find a perfectly profitable property, tracking it down without information such as listing_url may make it harder to determine which specific property we are looking at.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [15]:
correlation_matrix = df.corr(numeric_only=True)

# Get correlations with 'price'
price_correlations = correlation_matrix['price'].sort_values(ascending=False)

print("Correlation with Price (Top 10 positive and negative):")
print(price_correlations.head(11)) # Top 10 positive + price itself
print(price_correlations.tail(10)) # Top 10 negative

# Optional: Visualize the correlation matrix (can be large for many features)
# plt.figure(figsize=(12, 10))
# sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
# plt.title('Correlation Matrix of Numeric Features')
# plt.show()

Correlation with Price (Top 10 positive and negative):
price                             1.000000
host_total_listings_count         0.746825
host_listings_count               0.437634
estimated_revenue_l365d           0.337275
calculated_host_listings_count    0.217856
maximum_nights                    0.201248
maximum_maximum_nights            0.131315
maximum_nights_avg_ntm            0.078577
accommodates                      0.067573
beds                              0.046365
bathrooms                         0.044801
Name: price, dtype: float64
number_of_reviews_ltm         -0.080146
availability_30               -0.083407
review_scores_accuracy        -0.096945
review_scores_communication   -0.103452
latitude                      -0.111249
review_scores_value           -0.112808
estimated_occupancy_l365d     -0.113693
availability_365              -0.117125
review_scores_rating          -0.118622
review_scores_checkin         -0.133001
Name: price, dtype: float64


### ✍️ Your Response: 🔧
1. It appears that the total host total listing count has the highest correlation with price. After that the correlation percentage drops off, but host listing count and estimated revenue in 365 days have the next highest correlation.

2. Host listing count (total and not total) both prove to be vitally useful and should be our primary focus after that if we are looking to predict price, maximum nights would be our next most useful "category" of variables to look at.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [16]:
# Drop non-numeric and high-cardinality categorical columns that are not suitable for direct use in linear regression
# Many of these were previously dropped but might still be present if the user didn't drop them all.
# We will focus on columns that are directly numeric or can be easily one-hot encoded after further cleaning.

# Identify non-numeric columns for initial exclusion or further processing
non_numeric_cols = df.select_dtypes(include=['object']).columns

# Drop columns that are IDs, URLs, or free text, and those with many missing values or no variance
columns_to_exclude_from_features = [
    'listing_url', 'scrape_id', 'last_scraped', 'source', 'neighborhood_overview',
    'picture_url', 'host_url', 'host_name', 'host_since', 'host_location',
    'host_response_time', 'host_response_rate', 'host_acceptance_rate',
    'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
    'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
    'neighbourhood', 'neighbourhood_cleansed', 'bathrooms_text',
    'amenities', 'has_availability', 'calendar_last_scraped', 'last_review',
    'instant_bookable' # This could be converted, but let's simplify for now
]

# Filter out columns that do not exist in the current DataFrame
columns_to_exclude_from_features = [col for col in columns_to_exclude_from_features if col in df.columns]

# Also exclude columns with all NaNs or a single unique value (if any are left)
for col in df.columns:
    if df[col].isnull().all() or df[col].nunique() == 1:
        if col not in columns_to_exclude_from_features:
            columns_to_exclude_from_features.append(col)

# Define target variable
y = df['price'].copy()

# Drop the target variable and other excluded columns from the features DataFrame
X = df.drop(columns=['price'] + columns_to_exclude_from_features, errors='ignore').copy()

# Handle missing values in numerical features with the mean
for col in X.select_dtypes(include=['float64', 'int64']).columns:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].mean())

# One-hot encode categorical features remaining in X
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)


# Drop rows where 'price' is NaN from both X and y
valid_price_indices = df['price'].dropna().index
X = X.loc[valid_price_indices]
y = y.loc[valid_price_indices]

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")
print("First 5 rows of X:")
print(X.head())
print("First 5 rows of y:")
print(y.head())

Shape of features (X): (1844, 74)
Shape of target (y): (1844,)
First 5 rows of X:
   host_id  host_listings_count  host_total_listings_count  latitude  \
2  4895584                  1.0                        5.0  32.73643   
3  7892516                  1.0                        1.0  33.02235   
4   826752                  2.0                        2.0  32.71192   
5   826752                  2.0                        2.0  32.71330   
6  9951593                  3.0                        5.0  32.72627   

   longitude  accommodates  bathrooms  bedrooms  beds  minimum_nights  ...  \
2  -97.38305             2        1.0       0.0   2.0              30  ...   
3  -97.30032             2        1.0       1.0   1.0               1  ...   
4  -97.36889             2        1.0       2.0   5.0              30  ...   
5  -97.36832             2        1.0       1.0   1.0              30  ...   
6  -97.33481             2        1.0       1.0   1.0              29  ...   

   property_type

### ✍️ Your Response: 🔧
1. The majority of the columns that are included are those that are related to room type, location information, and other information regarding the property such as number of rooms and bathrooms.

2. This is a regression problem because we are looking for the variables that are most correlated with price. As we figure this out we can begin to build a training and sample dataset to determine future price. Essentially, because we are looking to predict price utilizing other variables this makes it a regression problem.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [17]:
# Split data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1475, 74)
X_test shape: (369, 74)
y_train shape: (1475,)
y_test shape: (369,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [18]:
# Initialize and fit the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict prices on the test set
y_pred = model.predict(X_test)

print("Model fitting and prediction complete.")

Model fitting and prediction complete.


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [19]:
# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)

# Calculate R-squared (R²)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

Mean Squared Error (MSE): 3752185.78
R-squared (R²): 0.83


### ✍️ Your Response: 🔧
1. We have an R^2 value of .83. Overall, this shows that we have a relatively strong correlation. Our model is fairly accurate when matched up with price and with an R^2 value of .83 this means that only 17% of our data's price variation is unexplained by model.

2. We have a very large MSE value of 3,752,185. Some ways that we can shrink this number is by scaling our data, creating a confidence interval, and potentially including additional variables; even some of the variables dropped earlier may reduce our MSE.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [20]:
# Create a DataFrame of coefficients
coefficients_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})

# Sort by the absolute value of coefficients to see the most impactful features
coefficients_df['Abs_Coefficient'] = abs(coefficients_df['Coefficient'])
coefficients_df = coefficients_df.sort_values(by='Abs_Coefficient', ascending=False)

print("Top 10 Most Impactful Features (by absolute coefficient value):")
print(coefficients_df.head(10))

print("\nBottom 10 Least Impactful Features (by absolute coefficient value):")
print(coefficients_df.tail(10))

Top 10 Most Impactful Features (by absolute coefficient value):
                                            Feature   Coefficient  \
71                             room_type_Hotel room  23064.616590   
51          property_type_Entire serviced apartment  -3123.683918   
56  property_type_Private room in bed and breakfast   2608.180076   
3                                          latitude  -2521.036634   
66              property_type_Private room in villa   2482.460387   
57           property_type_Private room in bungalow   2252.377395   
38      calculated_host_listings_count_shared_rooms  -2002.881799   
37     calculated_host_listings_count_private_rooms  -1849.368305   
55                       property_type_Private room   1836.900654   
62               property_type_Private room in loft   1815.853320   

    Abs_Coefficient  
71     23064.616590  
51      3123.683918  
56      2608.180076  
3       2521.036634  
66      2482.460387  
57      2252.377395  
38      2002.881799  


### ✍️ Your Response: 🔧
1. The room type at a hotel increases price the most as we have the largest coefficient out of all of our variables.

2. The most surprising negative variable that we have is the property type in apartments. This is contrary to the prior answer as you would expect that if the room type for a hotel has a high coefficient it isn't that far off to think that the property type for an apartment might also have a high coefficient.

3. This would be vitally useful not only to hotel owners but also apartment owners. To a hotel owner the room type means a huge amount which might lead the owner to either add more or less suites (depending on the preferred room type). And to an apartment owner, the property type means much less likely meaning that an apartment alone with few other factors is enough for most people.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [21]:
# 1. Choose your top 3–5 features with the strongest absolute coefficients
# From the previous output, let's select the top features based on Abs_Coefficient:
# 71                             room_type_Hotel room  23064.616590
# 51          property_type_Entire serviced apartment  -3123.683918
# 56  property_type_Private room in bed and breakfast   2608.180076
# 3                                          latitude  -2521.036634
# 66              property_type_Private room in villa   2482.460387

top_features = [
    'room_type_Hotel room',
    'property_type_Entire serviced apartment',
    'property_type_Private room in bed and breakfast',
    'latitude',
    'property_type_Private room in villa'
]

# Ensure these columns exist in X
actual_top_features = [f for f in top_features if f in X.columns]

if not actual_top_features:
    print("None of the selected top features were found in the DataFrame X. Please check feature names.")
else:
    print(f"Refining model with features: {actual_top_features}")

    # 2. Rebuild the regression model using just those features
    X_train_refined = X_train[actual_top_features]
    X_test_refined = X_test[actual_top_features]

    model_refined = LinearRegression()
    model_refined.fit(X_train_refined, y_train)
    y_pred_refined = model_refined.predict(X_test_refined)

    # 3. Compare MSE and R² between the baseline and refined model
    mse_refined = mean_squared_error(y_test, y_pred_refined)
    r2_refined = r2_score(y_test, y_pred_refined)

    print(f"\nRefined Model Performance:")
    print(f"  Mean Squared Error (MSE): {mse_refined:.2f}")
    print(f"  R-squared (R²): {r2_refined:.2f}")

    print(f"\nBaseline Model Performance:")
    print(f"  Mean Squared Error (MSE): {mse:.2f}")
    print(f"  R-squared (R²): {r2:.2f}")


Refining model with features: ['room_type_Hotel room', 'property_type_Entire serviced apartment', 'property_type_Private room in bed and breakfast', 'latitude', 'property_type_Private room in villa']

Refined Model Performance:
  Mean Squared Error (MSE): 2561836.24
  R-squared (R²): 0.88

Baseline Model Performance:
  Mean Squared Error (MSE): 3752185.78
  R-squared (R²): 0.83


### ✍️ Your Response: 🔧
1. Those that were kept had the greatest coefficients. These were kept because the high coefficients likely has the greatest impact on our data in regards to price.

2. Looking at our refined model performance we can see that our model did improve. Given the amount isn't extreme; our R^2 value improved from .83 to .88.

3. If we are looking for variables that purely explain price I recommend that we use the refined model. But if we want to conduct further analysis outside of price I would recommend that we use the prior model as more variables are included in it.

4. Being able to predict specific prices or trends by focusing on only the most specific data aligns with my goals of predictive analysis. If we can find the most useful variables and track those over time then apply those to a model, it can really help us track future revenues, expenses, etc.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. Our model helped us answer, "What factors play the biggest role in regards to price?"

2. I would recommend that Airbnb and hosts really focuses on the room types when pricing out rooms. Certain room types are going to be more expensive than others almost regardless of what property you are looking at. This should be the primary focus for hosts when pricing out and the focus of Airbnb when expecting future revenues.

3. The next step now that we have identified that room type is the most correlated with price is to break each property out by room price and find out which one of those then has the highest coefficient. By doing so we can figure out revenues, expenses and other very important accounting and financial information.

4. This is correlated to my future goal of leadership and getting an MBA overall. Leadership likely is going to have to figure out what is making them the most money and what is most profitable for the business. Looking at things holistically and then slowly "zooming" in on data that explains what we are looking for the most is vitally important when figuring out where to allocate resources and guide the overall goals of the business.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [22]:
!jupyter nbconvert --to html "assignment_11_PetersonSteven.ipynb"

[NbConvertApp] Converting notebook assignment_11_PetersonSteven.ipynb to html
[NbConvertApp] Writing 361966 bytes to assignment_11_PetersonSteven.html
